# Gold Annotation Comparison Legacy Parcor (Row-Level)

This notebook applies the row-level parcor evaluation protocol (from
`Gold_Annotation_Comparison.ipynb`) to the **legacy** gold dataset. Together
with the new-gold version, it extends row-level parcor evaluation coverage
from 5 dicts to 12 dicts (5 new + 7 legacy).

## How this differs from the new-gold version

| Aspect | New gold | Legacy gold (this notebook) |
|---|---|---|
| Gold input | A1 + A2 template CSVs | Tahap 1 + Tahap 2 xlsx files (merged) |
| Reconciliation | Required (A1 vs A2) | Not needed (single-keyed) |
| IAA metrics | Reported (κ, AC1, etc.) | Not applicable |
| POS comparison | Reported | Not applicable (no POS column in legacy gold) |
| Lemma sample size | Fixed at 100 per dict | Variable (~30-260 per dict, ~461 with parcor pairs total) |
| Match levels | strict / normalized / fuzzy | Same |
| Outcome classification | 5-way (TP / wrong_extraction / missed_lemma / FP / TN) | Same |
| Concatenation detection | Yes | Yes |

The methodology asymmetry (single-keyed vs double-keyed) is preserved in
reporting: results from this notebook should be interpreted as lower-rigor
than the new-gold results, since there's no agreement validation between
annotators. Both are useful: new gold gives rigorous narrow coverage; legacy
gold gives less rigorous but broader coverage.

## Parameterized run

This notebook is parameterized over three dimensions (cell 1):

- **ALGO**: `"legacy"` (original extraction algorithm) or `"new"` (new algorithm)
- **STAGE**: `"fixed"` (post-fix-script outputs) or `"spellchecked"` (post-spellcheck outputs)
- **STRATEGY**: `None` when `STAGE="fixed"`; one of `"original"`, `"B"`, `"C"` when `STAGE="spellchecked"`

Outputs go to `evaluationCSV/evaluation_legacy_gold_parcor/<condition>/`.

## Outputs

For each of the 7 legacy dicts (those with parcor gold pairs):
- `<dict_id>_comparison_A_vs_C.csv`: row-by-row comparison
- `<dict_id>_comparison_summary.csv`: per-dict precision/recall/F1 at three match levels

Plus a cross-dict summary:
- `_combined_summary.csv`

## 1. Configuration

In [36]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
from collections import Counter

# ===========================================================================
# RUN CONFIGURATION — edit these three variables, then run all cells
# ===========================================================================
ALGO     = "legacy"   # "legacy" or "new"
STAGE    = "fixed"    # "fixed" or "spellchecked"
STRATEGY = None       # None when STAGE="fixed"; "original" | "B" | "C" when "spellchecked"
# ===========================================================================

# Validate config
assert ALGO in ("legacy", "new"), f"ALGO must be 'legacy' or 'new', got {ALGO!r}"
assert STAGE in ("fixed", "spellchecked"), f"STAGE must be 'fixed' or 'spellchecked', got {STAGE!r}"
if STAGE == "fixed":
    assert STRATEGY is None, "STRATEGY must be None when STAGE='fixed'"
else:
    assert STRATEGY in ("original", "B", "C"), \
        f"STRATEGY must be 'original'|'B'|'C' when STAGE='spellchecked', got {STRATEGY!r}"

# === Legacy dicts with potential parcor gold ===
LEGACY_DICTS = ["18", "34", "42", "54", "68", "71", "89"]

# === Legacy gold files ===
GOLD_DIR = Path("../csvAnalysis/legacy_gold_diagnostic")
GOLD_TAHAP1 = GOLD_DIR / "GoldEntries_Tahap1_Merged.xlsx"
GOLD_TAHAP2 = GOLD_DIR / "GoldEntries_Tahap2_Merged.xlsx"

# === Resolve pipeline parcor input from config ===
EKSTRAKSI_ROOT = Path("../Ekstraksi")
_NEW_INFIX = " (New)" if ALGO == "new" else ""

if STAGE == "fixed":
    PARCOR_DIR = EKSTRAKSI_ROOT / f"11.{_NEW_INFIX} Parallel Corpus - Fixed"
    PARCOR_FILENAME_TPL = "{dict_id}_Parcor_audit.csv"
else:
    PARCOR_DIR = EKSTRAKSI_ROOT / f"12.{_NEW_INFIX} Parallel Corpus - Spellcheck Detection" / f"dict_strategy_{STRATEGY}"
    PARCOR_FILENAME_TPL = "{dict_id}_Parcor_spellcheck.csv"

# === Match thresholds ===
FUZZY_THRESHOLD = 0.75
CONCAT_LENGTH_RATIO = 2.0
CONCAT_MIN_SENTENCE_ENDS = 2

# === Output directory derived from config ===
_stage_label = "fixed" if STAGE == "fixed" else f"spellcheck_{STRATEGY}"
_condition = f"{ALGO}_algo_{_stage_label}"
OUT_DIR = Path("../evaluationCSV/evaluation_legacy_gold_parcor") / _condition
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"=== Configuration ===")
print(f"  ALGO:     {ALGO}")
print(f"  STAGE:    {STAGE}")
print(f"  STRATEGY: {STRATEGY}")
print(f"\n=== Resolved paths ===")
print(f"  Parcor dir: {PARCOR_DIR}")
print(f"  Gold:       {GOLD_TAHAP1.parent}")
print(f"  OUT_DIR:    {OUT_DIR.resolve()}")

=== Configuration ===
  ALGO:     legacy
  STAGE:    fixed
  STRATEGY: None

=== Resolved paths ===
  Parcor dir: ..\Ekstraksi\11. Parallel Corpus - Fixed
  Gold:       ..\csvAnalysis\legacy_gold_diagnostic
  OUT_DIR:    C:\Users\Legion\OneDrive\Documents\UNI\TA\tugas-akhir-data-mining\TAEkstraksiKamus\evaluationCSV\evaluation_legacy_gold_parcor\legacy_algo_fixed


## 2. Load and consolidate legacy gold parcor pairs

We load both tahaps, concatenate per dict, and filter to rows that have a
parcor pair (both `kalimat_asal` and `kalimat_tujuan` non-empty). These are
the gold C entries we compare the pipeline output against.

In [37]:
def load_tahap(path, tahap_label):
    if not path.exists():
        print(f"  ⚠ Gold file missing: {path}")
        return pd.DataFrame()
    xl = pd.ExcelFile(path)
    frames = []
    for sheet in xl.sheet_names:
        df = pd.read_excel(path, sheet_name=sheet)
        df["dict_id"] = sheet
        df["tahap"] = tahap_label
        frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


gold_t1 = load_tahap(GOLD_TAHAP1, "1")
gold_t2 = load_tahap(GOLD_TAHAP2, "2")
gold_all = pd.concat([gold_t1, gold_t2], ignore_index=True)

# Normalize string columns
for col in ["kata_asal", "kalimat_asal", "kalimat_tujuan"]:
    if col in gold_all.columns:
        gold_all[col] = (gold_all[col].fillna("").astype(str)
                         .str.replace("\r", " ", regex=False)
                         .str.replace("\n", " ", regex=False)
                         .str.replace(r"\s+", " ", regex=True)
                         .str.strip())

# Filter to rows with parcor pairs (both kalimat columns non-empty)
gold_parcor = gold_all[
    (gold_all["kalimat_asal"] != "") &
    (gold_all["kalimat_tujuan"] != "")
].copy()

# Restrict to legacy dicts
gold_parcor = gold_parcor[gold_parcor["dict_id"].isin(LEGACY_DICTS)].copy()

print(f"Total legacy gold rows: {len(gold_all)}")
print(f"Rows with parcor pairs: {len(gold_parcor)}")
print(f"\nPer-dict parcor row counts:")
print(gold_parcor.groupby("dict_id").size().to_string())

Total legacy gold rows: 1021
Rows with parcor pairs: 600

Per-dict parcor row counts:
dict_id
34      1
42     22
54    141
68    168
71    126
89    142


## 3. Normalization and similarity helpers

Same definitions as the new-gold comparison: strict, normalized, and fuzzy
(Jaccard on character trigrams) match levels.

In [38]:
_normalize_re = re.compile(r"\s+")

def normalize_text(s: str) -> str:
    """Lowercase, collapse internal whitespace, strip leading/trailing punctuation."""
    if not s:
        return ""
    s = s.lower().strip()
    s = _normalize_re.sub(" ", s)
    s = s.strip(" .,;:!?\"'()[]{}")
    return s


def char_ngrams(s: str, n: int = 3) -> set:
    s_norm = normalize_text(s)
    if len(s_norm) < n:
        return {s_norm} if s_norm else set()
    return {s_norm[i:i+n] for i in range(len(s_norm) - n + 1)}


def fuzzy_similarity(a: str, b: str) -> float:
    ga = char_ngrams(a)
    gb = char_ngrams(b)
    if not ga and not gb:
        return 1.0
    if not ga or not gb:
        return 0.0
    return len(ga & gb) / len(ga | gb)


def match_level(c_text: str, a_text: str) -> dict:
    strict = (c_text == a_text)
    normalized = (normalize_text(c_text) == normalize_text(a_text))
    fuzzy_score = fuzzy_similarity(c_text, a_text)
    fuzzy = fuzzy_score >= FUZZY_THRESHOLD
    return {
        "strict": strict,
        "normalized": normalized,
        "fuzzy": fuzzy,
        "fuzzy_score": round(fuzzy_score, 3),
    }


def detect_concatenation(a_text: str, c_text: str) -> bool:
    if not a_text:
        return False
    sentence_ends = sum(a_text.count(c) for c in ".!?")
    if sentence_ends >= CONCAT_MIN_SENTENCE_ENDS:
        return True
    if c_text and len(a_text) > CONCAT_LENGTH_RATIO * len(c_text):
        return True
    return False

## 4. Load pipeline parcor outputs per dict

In [39]:
def load_pipeline_parcor(dict_id: str) -> pd.DataFrame:
    """Load the pipeline parcor for this dict, or return empty DataFrame if missing."""
    filename = PARCOR_FILENAME_TPL.format(dict_id=dict_id)
    path = PARCOR_DIR / filename
    if not path.exists():
        return pd.DataFrame()
    df = pd.read_csv(path)
    for c in df.columns:
        if df[c].dtype == object:
            df[c] = (df[c].fillna("").astype(str)
                     .str.replace("\r", " ", regex=False)
                     .str.replace("\n", " ", regex=False)
                     .str.replace(r"\s+", " ", regex=True)
                     .str.strip())
    return df


pipeline_per_dict = {}
print("=== Pipeline parcor availability per dict ===")
for did in LEGACY_DICTS:
    df = load_pipeline_parcor(did)
    pipeline_per_dict[did] = df
    print(f"  Dict #{did}: {len(df)} pipeline rows")

=== Pipeline parcor availability per dict ===
  Dict #18: 2341 pipeline rows
  Dict #34: 4407 pipeline rows
  Dict #42: 3683 pipeline rows
  Dict #54: 1174 pipeline rows
  Dict #68: 2044 pipeline rows
  Dict #71: 727 pipeline rows
  Dict #89: 935 pipeline rows


## 5. Per-dict row-level comparison

For each legacy gold parcor row, look up the matching lemma in the pipeline
parcor and classify the outcome (TP / wrong_extraction / missed_lemma / FP /
TN). FP is rare in this design because legacy gold doesn't include "no-pair"
or "not-found" rows, by construction we only loaded rows with parcor pairs.
But we keep the 5-outcome structure for parity with the new-gold notebook.

In [40]:
def precompute_pipeline_pairs(pipeline_df: pd.DataFrame):
    """Precompute pipeline pairs for content-keyed matching.

    Returns (normalized_pairs_set, raw_pairs_list).
    - normalized_pairs_set: set of (norm_asal, norm_tuj) tuples for fast O(1) lookup
    - raw_pairs_list: list of (asal, tuj) tuples for strict and fuzzy comparison

    Schema-tolerant: tries common column-name variants for the asal/tuj columns.
    """
    if len(pipeline_df) == 0:
        return set(), []

    asal_col = next((c for c in ["kalimat_asal", "kalimat_asal_original"]
                     if c in pipeline_df.columns), None)
    tuj_col  = next((c for c in ["kalimat_tujuan", "kalimat_tujuan_original"]
                     if c in pipeline_df.columns), None)
    if not asal_col or not tuj_col:
        return set(), []

    norm_pairs = set()
    raw_pairs = []
    for _, row in pipeline_df.iterrows():
        asal = str(row[asal_col]).strip() if pd.notna(row[asal_col]) else ""
        tuj  = str(row[tuj_col]).strip()  if pd.notna(row[tuj_col])  else ""
        if asal and tuj:
            norm_pairs.add((normalize_text(asal), normalize_text(tuj)))
            raw_pairs.append((asal, tuj))
    return norm_pairs, raw_pairs


def find_matching_pair(c_asal: str, c_tuj: str, norm_pairs: set, raw_pairs: list):
    """Find if (c_asal, c_tuj) appears as a pair in the pipeline output."""
    if not c_asal or not c_tuj:
        return {
            "found_strict": False, "found_normalized": False, "found_fuzzy": False,
            "best_a_asal": "", "best_a_tuj": "",
            "best_asal_fuzzy_score": 0.0, "best_tuj_fuzzy_score": 0.0,
        }

    # Normalized lookup (fast, O(1))
    c_norm = (normalize_text(c_asal), normalize_text(c_tuj))
    found_normalized = c_norm in norm_pairs

    # Strict lookup
    found_strict = any(a == c_asal and b == c_tuj for a, b in raw_pairs)

    # Fuzzy: scan all pipeline pairs, find best by min(asal_sim, tuj_sim)
    best_score = 0.0
    best_pair = ("", "")
    best_asal_score = 0.0
    best_tuj_score = 0.0
    for a_asal, a_tuj in raw_pairs:
        s_asal = fuzzy_similarity(c_asal, a_asal)
        s_tuj  = fuzzy_similarity(c_tuj, a_tuj)
        combined = min(s_asal, s_tuj)
        if combined > best_score:
            best_score = combined
            best_pair = (a_asal, a_tuj)
            best_asal_score = s_asal
            best_tuj_score = s_tuj
    found_fuzzy = best_score >= FUZZY_THRESHOLD

    return {
        "found_strict":          found_strict,
        "found_normalized":      found_normalized,
        "found_fuzzy":           found_fuzzy,
        "best_a_asal":           best_pair[0],
        "best_a_tuj":            best_pair[1],
        "best_asal_fuzzy_score": round(best_asal_score, 3),
        "best_tuj_fuzzy_score":  round(best_tuj_score, 3),
    }


def compare_dict(dict_id: str) -> pd.DataFrame:
    """Content-keyed row-level comparison for one dict.

    For each gold parcor pair, check whether that pair appears anywhere in the
    pipeline output (matching by content, not by lemma). This is the same logic
    used in Gold_Annotation_Comparison.ipynb (new-gold notebook).
    """
    gold = gold_parcor[gold_parcor["dict_id"] == dict_id].copy().reset_index(drop=True)
    if len(gold) == 0:
        return pd.DataFrame()

    pipeline_df = pipeline_per_dict[dict_id]
    norm_pairs, raw_pairs = precompute_pipeline_pairs(pipeline_df)

    rows = []
    for _, g in gold.iterrows():
        lemma = g["kata_asal"]
        c_asal = g["kalimat_asal"]
        c_tuj  = g["kalimat_tujuan"]

        # All gold rows have parcor pairs here (filtered upstream in Cell 4)
        c_has_pair = True

        match = find_matching_pair(c_asal, c_tuj, norm_pairs, raw_pairs)
        a_asal = match["best_a_asal"]
        a_tuj  = match["best_a_tuj"]
        is_concat = detect_concatenation(a_asal, c_asal) or detect_concatenation(a_tuj, c_tuj)

        # Outcome under content-keyed comparison (primary metric: normalized)
        # Simplified to TP/FN since legacy gold filter guarantees c_has_pair=True
        if match["found_normalized"]:
            outcome = "TP"
        else:
            outcome = "FN"

        rows.append({
            "dict_id":                  dict_id,
            "kata_asal":                lemma,
            "c_has_pair":               c_has_pair,
            "c_kalimat_asal":           c_asal,
            "c_kalimat_tujuan":         c_tuj,
            "best_a_kalimat_asal":      a_asal,
            "best_a_kalimat_tujuan":    a_tuj,
            "match_strict":             match["found_strict"],
            "match_normalized":         match["found_normalized"],
            "match_fuzzy":              match["found_fuzzy"],
            "best_asal_fuzzy_score":    match["best_asal_fuzzy_score"],
            "best_tuj_fuzzy_score":     match["best_tuj_fuzzy_score"],
            "concatenation_candidate":  is_concat,
            "outcome":                  outcome,
        })
    return pd.DataFrame(rows)


all_comparisons = {}
for did in LEGACY_DICTS:
    df = compare_dict(did)
    if len(df) == 0:
        print(f"  Dict #{did}: no legacy parcor gold rows to evaluate")
        continue
    out_path = OUT_DIR / f"{did}_comparison_A_vs_C.csv"
    df.to_csv(out_path, index=False)
    all_comparisons[did] = df
    counts = Counter(df["outcome"])
    print(f"  Dict #{did}: {len(df)} rows | TP={counts.get('TP', 0)} "
          f"FN={counts.get('FN', 0)} concat={df['concatenation_candidate'].sum()}")


  Dict #18: no legacy parcor gold rows to evaluate
  Dict #34: 1 rows | TP=0 FN=1 concat=0
  Dict #42: 22 rows | TP=0 FN=22 concat=0
  Dict #54: 141 rows | TP=10 FN=131 concat=6
  Dict #68: 168 rows | TP=29 FN=139 concat=3
  Dict #71: 126 rows | TP=48 FN=78 concat=5
  Dict #89: 142 rows | TP=105 FN=37 concat=0


## 6. Per-dict and combined summary

Compute precision/recall/F1 at three match levels per dict. Combine across
all dicts for a cross-dict comparison.

In [41]:
def compute_metrics(df):
    """Under content-keyed comparison: only recall is well-defined.

    For each match level (strict/normalized/fuzzy):
    - TP = gold pair found at that match level in pipeline output
    - FN = gold pair not found

    All gold rows in legacy parcor evaluation have c_has_pair=True (filtered
    upstream), so there are no FP/TN cases. Precision is undefined under
    content-keyed comparison.
    """
    metrics = {}
    for level in ["strict", "normalized", "fuzzy"]:
        tp = fn = 0
        for _, row in df.iterrows():
            m = row[f"match_{level}"]
            if m:
                tp += 1
            else:
                fn += 1
        recall = tp / (tp + fn) if (tp + fn) else float("nan")
        metrics[level] = {
            "tp":        tp,
            "fn":        fn,
            "tn":        0,
            "fp":        float("nan"),  # undefined under content-keyed
            "precision": float("nan"),  # undefined under content-keyed
            "recall":    round(recall, 4) if not pd.isna(recall) else float("nan"),
            "f1":        float("nan"),  # undefined under content-keyed
        }
    return metrics


combined_rows = []
for did, df in all_comparisons.items():
    metrics = compute_metrics(df)
    for level, m in metrics.items():
        combined_rows.append({
            "dict_id":     did,
            "n_gold_rows": len(df),
            "match_level": level,
            **m,
        })
    # Per-dict summary
    per_dict_rows = []
    for level, m in metrics.items():
        per_dict_rows.append({"match_level": level, **m})
    summary_path = OUT_DIR / f"{did}_comparison_summary.csv"
    pd.DataFrame(per_dict_rows).to_csv(summary_path, index=False)

combined_df = pd.DataFrame(combined_rows)
combined_path = OUT_DIR / "_combined_summary.csv"
combined_df.to_csv(combined_path, index=False)

print("=== Combined summary (legacy gold parcor, content-keyed) ===")
print(combined_df.to_string(index=False))
print()
print("Note: under content-keyed comparison, precision and F1 are undefined")
print("because we cannot define 'false positives' without a per-lemma expectation.")
print("Recall is the primary metric.")
print(f"\nWritten: {combined_path.name}")


=== Combined summary (legacy gold parcor, content-keyed) ===
dict_id  n_gold_rows match_level  tp  fn  tn  fp  precision  recall  f1
     34            1      strict   0   1   0 NaN        NaN  0.0000 NaN
     34            1  normalized   0   1   0 NaN        NaN  0.0000 NaN
     34            1       fuzzy   0   1   0 NaN        NaN  0.0000 NaN
     42           22      strict   0  22   0 NaN        NaN  0.0000 NaN
     42           22  normalized   0  22   0 NaN        NaN  0.0000 NaN
     42           22       fuzzy   0  22   0 NaN        NaN  0.0000 NaN
     54          141      strict  10 131   0 NaN        NaN  0.0709 NaN
     54          141  normalized  10 131   0 NaN        NaN  0.0709 NaN
     54          141       fuzzy  28 113   0 NaN        NaN  0.1986 NaN
     68          168      strict   6 162   0 NaN        NaN  0.0357 NaN
     68          168  normalized  29 139   0 NaN        NaN  0.1726 NaN
     68          168       fuzzy  47 121   0 NaN        NaN  0.2798 NaN
   

## 7. Final readout

In [42]:
print("=" * 60)
print(f"  Legacy Gold Parcor — Row-Level Evaluation (content-keyed)")
print("=" * 60)
print(f"\n  Condition:   ALGO={ALGO}, STAGE={STAGE}, STRATEGY={STRATEGY}")
print(f"  Output dir:  {OUT_DIR.resolve()}")
print()

print(f"  === Dict-level normalized recall ===")
for did in LEGACY_DICTS:
    if did not in all_comparisons:
        continue
    rows = combined_df[(combined_df["dict_id"] == did) & (combined_df["match_level"] == "normalized")]
    if len(rows):
        r = rows.iloc[0]
        print(f"    Dict #{did}: n={r['n_gold_rows']}, TP={r['tp']}, FN={r['fn']}, "
              f"recall={r['recall']}")
print()
print(f"  Note: precision and F1 are undefined under content-keyed comparison;")
print(f"  only recall is reported as the primary metric.")
print(f"  See `_combined_summary.csv` for full metrics at all three match levels.")


  Legacy Gold Parcor — Row-Level Evaluation (content-keyed)

  Condition:   ALGO=legacy, STAGE=fixed, STRATEGY=None
  Output dir:  C:\Users\Legion\OneDrive\Documents\UNI\TA\tugas-akhir-data-mining\TAEkstraksiKamus\evaluationCSV\evaluation_legacy_gold_parcor\legacy_algo_fixed

  === Dict-level normalized recall ===
    Dict #34: n=1, TP=0, FN=1, recall=0.0
    Dict #42: n=22, TP=0, FN=22, recall=0.0
    Dict #54: n=141, TP=10, FN=131, recall=0.0709
    Dict #68: n=168, TP=29, FN=139, recall=0.1726
    Dict #71: n=126, TP=48, FN=78, recall=0.381
    Dict #89: n=142, TP=105, FN=37, recall=0.7394

  Note: precision and F1 are undefined under content-keyed comparison;
  only recall is reported as the primary metric.
  See `_combined_summary.csv` for full metrics at all three match levels.
